## 평균, 실자료. 평균 키 by gender,
 남-여 평균 키
 NCD-RisC가 2020년 영국 의학 학술지 랜싯에 발표한 추정치에 따르면 2019년 기준 19세 일본인의 평균 신장은 남성 172.1㎝, 여성 158.5㎝였다. 같은 나이 한국인은 남성 175.5㎝, 여성 163.2㎝, 중국인은 남성 175.7㎝·여성 163.5㎝로 한국인과 중국인의 평균 신장이 남성의 경우 약 3㎝, 여성은 약 5㎝ 일본인보다 각각 컸다.

일본인의 키는 1896년부터 100년 동안 남성은 약 14.6㎝, 여성은 약 16㎝ 커졌지만 1990년대 이후에는 큰 변화가 없었다. 최근 자료에서도 일본의 17세 남학생 평균 키는 170.8㎝로, 과거 최고치인 170.9㎝와 거의 차이가 없다.
https://n.news.naver.com/mnews/article/277/0005818914 (아시아경제, 9/20/2026)

In [64]:
# One sample exercise
# 평균 추론. muhat, se, 신뢰구간, 검정통계량, pvalue.

import os
import numpy as np                          # numpy 라이브러리 전체.
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네.
import scipy as sci
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈. 필요한 통계 모듈만.
import statsmodels.api as sm                # 방대한 모델이라 개발자들이 많이 쓰는 모델 묶음.
from statsmodels.tsa import stattools       # time series analysis
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭. 소문자 못읽어.
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.weightstats import DescrStatsW, ztest
from scipy.stats import ttest_1samp


### 자료셋 정리
 gender, height

In [65]:
# 1. 데이터 준비. 읽기. 생성.
# 데이터 파일 읽기. github.com
dat_url = 'https://github.com/bahn28/gamja/blob/main/cs_nns_gndr_hgt.csv?raw=true'
df_dat = pd.read_csv(dat_url)
df_dat #.head()


,i,gender,ht
0,1,1,159.9
1,2,2,157.5
2,3,2,158.0
3,4,2,154.2
4,5,1,163.3
...,...,...,...
20123,20124,2,157.3
20124,20125,1,175.9
20125,20126,1,175.6
20126,20127,2,158.0


#### 변수 생성, 리네임, 일부 추출
2만 관측점의 일부만 사용.

추출 비율 조정 n_frac,

키, 성별 선택.
ht_1 남, ht_2 여

In [66]:
n_ttl = len(df_dat)
n_frac = 0.05                    # (0.025, 2.5%, 대략 500개)
n_smpld = int( n_frac * n_ttl)   # 추출 갯수, 비율,

# 단순 무작위 추출 (2.5%, 대략 500개)
df_smpl = df_dat.sample(n_smpld, replace=False, random_state=42)

df_smpl['ht_1']= df_smpl['ht'][df_smpl['gender']==1] #.to_numpy()  # male
df_smpl['ht_2']= df_smpl['ht'][df_smpl['gender']==2] #.to_numpy()  # female
hgt = df_smpl['ht_2'].dropna()    # choose male/female for analysis.
df_smpl


,i,gender,ht,ht_1,ht_2
15561,15562,1,170.9,170.9,NaN
13056,13057,2,144.4,NaN,144.4
5702,5703,1,170.5,170.5,NaN
3062,3063,2,158.5,NaN,158.5
9199,9200,2,155.1,NaN,155.1
...,...,...,...,...,...
17202,17203,2,161.1,NaN,161.1
4296,4297,2,165.2,NaN,165.2
878,879,1,167.5,167.5,NaN
10117,10118,2,158.5,NaN,158.5


### 표본평균의 분포
\begin{align}
 \hat \mu & \sim N \left(\mu, SE^2 \right) \\
 \widehat{SE} & = \sqrt{ \sigma^2 \over n }
\end{align}

### 가설검정, 검정통계량
\begin{align}
 H_0  : \mu  = \mu_0 \quad vs. \quad
 H_A  : \mu  \ne 0  
\end{align}

$$
 T_0 = { \hat\mu - \mu_0 \over \widehat{ SE } }
$$



#### 유의수준, 임계치

In [67]:
# 분포 임계치, 양방향, 우측값.
alpha = 0.05                          # significance level, two side.
zcv_r = stats.norm.ppf( 1- alpha/2 )  # critical value on the right, two side
tcv_r = stats.t.ppf(1 - alpha / 2, df= n_smpld-1 )
print( alpha, zcv_r, tcv_r )

0.05 1.959963984540054 1.9623272504292877


### 평균 키(표본평균) $ \hat \mu , \bar X $
$\hat \mu = { \sum X_i \over n }$

In [68]:
# 평균 키 추정치:
# 추정치: 평균, 표준오차, overall
muhat = hgt.mean()
print("표본평균 추정치: " , muhat )


표본평균 추정치:  157.68455882352941


### 평균 키(표본평균)의 분산, 표준오차
$ V = SE^2 $, $ \ SE = \sqrt{ \sigma^2 \over n } $

In [69]:

sig2_hat = hgt.var(ddof=1)
print("표본분산 추정치: " , sig2_hat )

sig_hat = ( sig2_hat  )**0.5
print("표준편차 추정치: " , sig_hat )

var_muhat = sig2_hat / n_smpld
print(" 표본평균의 분산 추정치: " , var_muhat )

#se_muhat = var_muhat**0.5
se_muhat = hgt.std(ddof=1) / np.sqrt( n_smpld )
print(" 표본평균의 표준오차 추정치: " , se_muhat )


표본분산 추정치:  35.013941609793086
표준편차 추정치:  5.917257946869739
 표본평균의 분산 추정치:  0.03480511094412832
 표본평균의 표준오차 추정치:  0.18656127932700375


### 평균추론 1. 신뢰구간
모평균의  
$ 100(1-\alpha) \% $ 신뢰구간
\begin{align}  
\hat \mu  & \pm z_{\alpha/2} \times \widehat{ SE }  \\  
\hat \mu  & \pm t_{\alpha/2, df} \times \widehat{ SE }  
\end{align}


In [70]:
# 신뢰구간, right, left,
ci_r = muhat + zcv_r * se_muhat
ci_l = muhat - zcv_r * se_muhat

print( f"   muhat,        se_muhat,              {100*(1-alpha)}%   confidence interval  ")
print(  muhat, se_muhat,   ci_l, ci_r )

print(" t분포를 적용하면 약간 차이 남")
ci_tr = muhat + tcv_r * se_muhat
ci_tl = muhat - tcv_r * se_muhat
print(  muhat, se_muhat,   ci_tl, ci_tr )

   muhat,        se_muhat,              95.0%   confidence interval  
157.68455882352941 0.18656127932700375 157.31890543513876 158.05021221192007
 t분포를 적용하면 약간 차이 남
157.68455882352941 0.18656127932700375 157.3184645412311 158.05065310582773


### 평균추론 2. 검정통계량, $ T_0 $
$$
 T_0 = { \hat\mu - \mu_0 \over \widehat{ SE } }
$$


#### 귀무가설, 대립가설
$ H_0 : \mu = \mu_0 $ vs. $ H_A: \mu \ne \mu_0 $

\begin{align}
 \text{reject } H_0
 & \quad
 \text{ if } |T_0| > z_{\alpha/2} \\
 & \quad
 \text{ if  p-value } < \alpha
\end{align}

In [71]:
# 가설검정. 전체, 톨
# 귀무가설 H0: mu = mu0
# 검정통계치, pvalue

mu_zero = 160
print("H0: mu = ", mu_zero, "HA: mu is not ", mu_zero)


H0: mu =  160 HA: mu is not  160


#### 검정통계치, 가설검정

In [72]:
t_0 = np.abs( ( muhat - mu_zero ) / se_muhat )
print("t_0= ", t_0 )
print( f"reject H0 if test statistic, t_0 = {t_0} > critical value zcv_r = {zcv_r} at significance level {alpha}" )
result_test = " 'Reject H0' " if t_0 > zcv_r else " 'Fail to reject H0' "   # 이건 되는 군.
print(f" 검정통계치,    임계치(유의수준={alpha}),    검정결과 ")
print( t_0 , zcv_r, result_test)

t_0=  12.411156188589867
reject H0 if test statistic, t_0 = 12.411156188589867 > critical value zcv_r = 1.959963984540054 at significance level 0.05
 검정통계치,    임계치(유의수준=0.05),    검정결과 
12.411156188589867 1.959963984540054  'Reject H0' 


### 평균추론 3. $ p $값
$ p$ value $= 2\times P(T_0 > |t_0|) $

In [73]:
# 이제 p값

p_val = 2*( 1 - stats.norm.cdf( t_0 ) )
p_val_t = 2*( 1 - stats.t.cdf( t_0, df=len(df_smpl)-1 ) )
p_val_sf = 2 * stats.t.sf(t_0, df=len(df_smpl)-1)

print(f" p값 = { p_val } " )
print(f" p값 = { p_val_t } " )
print(f" p값 = { p_val_sf } " )


f"reject H0 if p값 < {alpha}"


 p값 = 0.0 
 p값 = 0.0 
 p값 = 5.1976420254612976e-33 


'reject H0 if p값 < 0.05'

### 모듈로 확인, confirm

In [74]:
# # 모듈 이용 statsmodels.stats.weightstats
# # 신뢰구간,
# 차이가 나. 자유도 조정 안 하는 모양.
#    from statsmodels.stats.weightstats import DescrStatsW
ds_hgt = DescrStatsW( hgt )               # 모듈이 돌려준 객체
ci_ds = ds_hgt.tconfint_mean(alpha=0.05)   # 이건 tuple
#print(ci_ds)
#print(len(hgt)) #.df_denom)

print(" 모듈 DescrStatsW 반환 값  ")
print(f" 표본평균 : {ds_hgt.mean:.8f}")
print(f" 표본분산 : {ds_hgt.var:.8f}")
print(f" 표준편차 : {ds_hgt.std:.8f}")
print(f" 표준오차 : {(ds_hgt.std/np.sqrt(n_smpld )):.8f}")
print(" 신뢰구간 :", ", ".join(f"{x:.8f}" for x in ci_ds ))

print("# 차이가 나. DescrStatsW 자유도 조정 방식 ")
print("# var, n. std, n-1. 방식이 다르다고 하네.")
print("# 자유도 조정 후 계산값")
var_df = ds_hgt.var*n_smpld/(n_smpld-1)
std_df = var_df**0.5
se_df = std_df / np.sqrt( n_smpld )
print(f"# 표본분산 : {var_df:.8f} ")
print(f"# 표준편차 :  {std_df:.8f} ")
print(f"# 표준오차 :  {se_df:.8f}")

#print(" 신뢰구간 :", ", ".join(f"{x:.8f}" for x in ci_a ))

# 평균 t 검정. 모듈이 여럿임.
print(" 모듈 DescrStatsW 반환, ttest_mean() 적용 ")
print(" H0: mu =", mu_zero )
t0_ds, pval_ds, df_ds = ds_hgt.ttest_mean(mu_zero) # mu_zero겠지?

print(" 검정통계치 ",  t0_ds , df_ds )
print(" p-value  ",  pval_ds  )
# pvalu의 미세한 차이는 t.cdf()와 t.sf() 차이네.

 모듈 DescrStatsW 반환 값  
 표본평균 : 157.68455882
 표본분산 : 34.94957775
 표준편차 : 5.91181679
 표준오차 : 0.18638973
 신뢰구간 : 157.18620449, 158.18291316
# 차이가 나. DescrStatsW 자유도 조정 방식 
# var, n. std, n-1. 방식이 다르다고 하네.
# 자유도 조정 후 계산값
# 표본분산 : 34.98435345 
# 표준편차 :  5.91475726 
# 표준오차 :  0.18648244
 모듈 DescrStatsW 반환, ttest_mean() 적용 
 H0: mu = 160
 검정통계치  -9.126677414822662 543.0
 p-value   1.3797668032547931e-18


## 2개 그룹 평균 비교는 별도